# Jaxifying My Current Code Base

I want this simulation code to be able to run as fast as possible, so I'm making this notebook to work on parallelization using the `jax` package. This package will allow me to use `jit` (just in time) compiling to significantly reduce run-times. By offloading computations onto my brand new 5070 \\[^-^]/, I should be able to cut execution times significantly.

The original class code is shown below for `laser.py`:

In [1]:
import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
from matplotlib import cm
class Laser:
    def __init__(self, mu_x, mu_y, z, mode = [0, 0], mode_type = "h", res = 5000, waist = 1, power = 100, chop = True, chop_threshold = 1e-3, wavelen = 1064e-9):
        self.power = power
        self.waist = waist
        self.resolution = res
        self.chop = chop
        self.chop_threshold = chop_threshold
        self.wl = wavelen
        self.xyz = self.make_rays(self.power, self.resolution, mu_x, mu_y, z, mode, mode_type)
        self.x, self.y, self.P = self.set_coords()
        return
    def set_coords(self):
        if self.chop == True:
            x = self.xyz[np.isnan(self.xyz[:, 2]) == False][:, 0]
            y = self.xyz[np.isnan(self.xyz[:, 2]) == False][:, 1]
            P = self.xyz[np.isnan(self.xyz[:, 2]) == False][:, 2]
        else:
            x = self.xyz[:, 0]
            y = self.xyz[:, 1]
            P = self.xyz[:, 2]
        return x, y, P
    def set_power(self, power):
        self.power = power
        return power
    def get_power(self):
        return self.power
    def dy(self, dy = 0):
        self.xyz = np.column_stack([self.xyz[:, 0].flat, (self.xyz[:, 1] + dy).flat, self.xyz[:, 2].flat])
        self.set_coords()
        return
    def w(self, z, n = 1):
        return self.waist * np.sqrt(1 + np.power(z / (np.pi * self.waist * 2 * n / self.wl), 2))
    def make_rays(self, power, res, mu_x, mu_y, z = 0, mode = [0, 0], mode_type = "H"):
        from scipy.special import hermite, genlaguerre
        X, Y = np.meshgrid(np.linspace(-self.waist * 10, self.waist * 10, res), np.linspace(-self.waist * 10, self.waist * 10, res))
        if mode_type.upper() == "h".upper():
            xyz = np.column_stack([X.flat, Y.flat, power * (self.waist / self.w(z)) ** 2 * (hermite(mode[0])(np.sqrt(2) * (np.array(X.flat) - mu_x) / self.w(z)) * np.exp(- (np.array(X.flat) - mu_x) ** 2 / self.w(z) ** 2)) ** 2 * (hermite(mode[1])(np.sqrt(2) * (np.array(Y.flat) - mu_y) / self.w(z)) * np.exp(- (np.array(Y.flat) - mu_y) ** 2 / self.w(z) ** 2)) ** 2])
        if mode_type.upper() == "l".upper():
            xyz = np.column_stack([X.flat, Y.flat, power * (2 * np.sqrt((np.array(X.flat) - mu_x) ** 2 + (np.array(Y.flat) - mu_y) ** 2) ** 2 / self.w(z) ** 2) ** mode[1] * (genlaguerre(mode[0], mode[1])(2 * np.sqrt((np.array(X.flat) - mu_x) ** 2 + (np.array(Y.flat) - mu_y) ** 2) ** 2 / self.w(z) ** 2)) ** 2 * np.cos(mode[1] * np.arctan((np.array(Y.flat) - mu_y) / (np.array(X.flat) - mu_x))) ** 2 * np.exp(-2 * np.sqrt((np.array(X.flat) - mu_x) ** 2 + (np.array(Y.flat) - mu_y) ** 2) ** 2 / self.w(z) ** 2)])
        xyz[:, 2][np.isnan(xyz[:, 2]) == False] = power * xyz[:, 2][np.isnan(xyz[:, 2]) == False] / np.sum(xyz[:, 2][np.isnan(xyz[:, 2]) == False])
        if self.chop == True:
            xyz[np.ix_(np.where(np.abs(xyz[:, 2]) < self.chop_threshold)[0], [2])] = np.nan
        return xyz
    def plot_surface(self, ax = None, figsize = (10, 10), alpha = 1, cmap = cm.coolwarm, antialiased = False, show = False, save = False, name = "", *args, **kwargs):
        if ax == None:
            fig = plt.figure(figsize = figsize)
            ax = fig.add_subplot(111, projection='3d')
        else:
            fig = plt.gcf()
        ax.plot_surface(*[self.xyz[:, i].reshape(int(np.sqrt(len(self.xyz[:, i]))), int(np.sqrt(len(self.xyz[:, i])))) for i in range(3)], alpha = alpha, cmap = cmap, antialiased = antialiased, *args, **kwargs)
        if save and (name != ""):
            fig.savefig("")
        if show:
            plt.show()
        return fig, ax
    def imshow(self, ax = None, dimensions = (1, 1), figsize = (10, 10), cmap = cm.coolwarm, save = False, origin = "lower", extent = [],  name = "", show = False):
        if ax == None:
            fig, ax = plt.subplots(*dimensions, figsize = figsize)
        else:
            fig = plt.gcf()
        if extent != []:
            ax.imshow(self.xyz[:, 2].reshape(int(np.sqrt(len(self.xyz[:, 2]))), int(np.sqrt(len(self.xyz[:, 2])))), cmap = cmap, origin = origin, extent = extent)
        else:
            ax.imshow(self.xyz[:, 2].reshape(int(np.sqrt(len(self.xyz[:, 2]))), int(np.sqrt(len(self.xyz[:, 2])))), cmap = cmap, origin = origin, extent = [np.min(self.xyz[:, 0][np.isnan(self.xyz[:, 2]) == False]), np.max(self.xyz[:, 0][np.isnan(self.xyz[:, 2]) == False]), np.min(self.xyz[:, 1][np.isnan(self.xyz[:, 2]) == False]), np.max(self.xyz[:, 1][np.isnan(self.xyz[:, 2]) == False])])
        if save and (name != ""):
            fig.savefig("")
        if show:
            plt.show()
        return fig, ax
    def __len__(self):
        return len(self.xyz[:, 0].flat)

### Laser Jaxification

I will attempt to jaxify this exact class in the code block below. This means shifting all array objects and calculations into the `jax.numpy` class instead of `numpy`. I will also have to rewrite the hermite and general laguerre functions using jax style because scipy will be incompatible with jax's compiling system. Once this is finished, I will do a direct comparison of the methods.

In [3]:
import os
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"
import matplotlib.pyplot as plt
from matplotlib import cm
from jax import jit
import jax.numpy as jnp
import jax
from jax import tree_util

@jit
def jax_hermite(n, x):
    def body_fun(i, val):
        h_prev, h_curr = val
        return h_curr, 2 * x * h_curr - 2 * i * h_prev
    _, hn = jax.lax.fori_loop(0, n, body_fun, (jnp.zeros_like(x), jnp.ones_like(x)))
    return hn

@jit
def jax_genlaguerre(n, alpha, x):
    init_state = (0, jnp.zeros_like(x), jnp.ones_like(x))
    
    def body_fun(state):
        i, l_prev, l_curr = state
        l_next = ((2 * i + 1 + alpha - x) * l_curr - (i + alpha) * l_prev) / (i + 1)
        return (i + 1, l_curr, l_next)
    _, _, ln = jax.lax.while_loop(
        lambda state: state[0] < n,
        body_fun,
        init_state
    )
    return ln
class laser:
    def __init__(self, mu_x = 0, mu_y = 0, z = 0, mode = [0, 0], mode_type = 0, res = 5000, waist = 1, power = 100, chop = True, chop_threshold = 1e-3, wavelen = 1064e-9):
        self.power = power
        self.waist = waist
        self.resolution = res
        self.chop = chop
        self.chop_threshold = chop_threshold
        self.wl = wavelen
        self.mode = mode
        self.mode_type = mode_type
        self.mu_x = mu_x
        self.mu_y = mu_y
        self.z = z
        self.xyz = jnp.array([])
        self.x, self.y, self.P = jnp.array([]), jnp.array([]), jnp.array([])
        self.xyz = self.make_rays()
        self.x, self.y, self.P = self.set_coords()
        return
    def set_coords(self):
        if self.chop == True:
            mask = jnp.abs(self.xyz[:, 2]) >= self.chop_threshold
            idx = jnp.where(mask, size=self.resolution * self.resolution, fill_value=0)[0]
            x = self.xyz[:, 0][idx]
            y = self.xyz[:, 1][idx]
            P = self.xyz[:, 2][idx]
        else:
            x = self.xyz[:, 0]
            y = self.xyz[:, 1]
            P = self.xyz[:, 2]
        return x, y, P
    def dy(self, dy = 0):
        xyz = jnp.column_stack([self.xyz[:, 0].ravel(), (self.xyz[:, 1] + dy).ravel(), self.xyz[:, 2].ravel()])
        children, aux = self._tree_flatten()
        new_children = (xyz, self.x, self.y, self.P)
        new_obj = self._tree_unflatten(aux, new_children)
        new_x, new_y, new_P = new_obj.set_coords()
        final_children = (xyz, new_x, new_y, new_P)
        return self._tree_unflatten(aux, final_children)
    def w(self, n = 1):
        return self.waist * jnp.sqrt(1 + jnp.power(self.z / (jnp.pi * self.waist * 2 * n / self.wl), 2))
    @jit
    def make_rays(self):
        lin = jnp.linspace(-self.waist * 10, self.waist * 10, self.resolution)
        X, Y = jnp.meshgrid(lin, lin)
        x = jnp.array(X.ravel()) - self.mu_x
        y = jnp.array(Y.ravel()) - self.mu_y
        if self.mode_type == 0:
            x_part = (jax_hermite(self.mode[0], jnp.sqrt(2) * x / self.w()) * jnp.exp(-x ** 2 / self.w() ** 2)) ** 2
            y_part = (jax_hermite(self.mode[1], jnp.sqrt(2) * y / self.w()) * jnp.exp(-y ** 2 / self.w() ** 2)) ** 2
            p = self.power * (self.waist / self.w()) ** 2
            P =  p * x_part * y_part
            xyz = jnp.column_stack([X.ravel(), Y.ravel(), P])
        if self.mode_type == 1:
            r = jnp.sqrt(x ** 2 + y ** 2)
            p = self.power * (2 * r ** 2 / self.w() ** 2) ** self.mode[1]
            P = p * (jax_genlaguerre(self.mode[0], self.mode[1], 2 * r ** 2 / self.w() ** 2)) ** 2 * jnp.cos(self.mode[1] * jnp.arctan(y / x)) ** 2 * jnp.exp(-2 * r ** 2 / self.w() ** 2)
            xyz = jnp.column_stack([X.ravel(), Y.ravel(), P])
        xyz = jnp.column_stack([X.ravel(), Y.ravel(), self.power * xyz[:, 2] / jnp.sum(xyz[:, 2])])
        if self.chop == True:
            xyz = jnp.column_stack([X.ravel(), Y.ravel(), xyz[:, 2].at[jnp.where(jnp.abs(xyz[:, 2]) < self.chop_threshold)].set(np.nan)])
        return xyz
    def plot_surface(self, ax = None, figsize = (10, 10), alpha = 1, cmap = cm.coolwarm, antialiased = False, show = False, save = False, name = "", *args, **kwargs):
        if ax == None:
            fig = plt.figure(figsize = figsize)
            ax = fig.add_subplot(111, projection='3d')
        else:
            fig = plt.gcf()
        ax.plot_surface(*[self.xyz[:, i].reshape(int(jnp.sqrt(len(self.xyz[:, i]))), int(jnp.sqrt(len(self.xyz[:, i])))) for i in range(3)], alpha = alpha, cmap = cmap, antialiased = antialiased, *args, **kwargs)
        if save and (name != ""):
            fig.savefig("")
        if show:
            plt.show()
        return fig, ax
    def imshow(self, ax = None, dimensions = (1, 1), figsize = (10, 10), cmap = cm.coolwarm, save = False, origin = "lower", extent = [],  name = "", show = False):
        if ax == None:
            fig, ax = plt.subplots(*dimensions, figsize = figsize)
        else:
            fig = plt.gcf()
        if extent != []:
            ax.imshow(self.xyz[:, 2].reshape(int(jnp.sqrt(len(self.xyz[:, 2]))), int(jnp.sqrt(len(self.xyz[:, 2])))), cmap = cmap, origin = origin, extent = extent)
        else:
            ax.imshow(self.xyz[:, 2].reshape(int(jnp.sqrt(len(self.xyz[:, 2]))), int(jnp.sqrt(len(self.xyz[:, 2])))), cmap = cmap, origin = origin, extent = [jnp.min(self.xyz[:, 0][jnp.isnan(self.xyz[:, 2]) == False]), jnp.max(self.xyz[:, 0][jnp.isnan(self.xyz[:, 2]) == False]), jnp.min(self.xyz[:, 1][jnp.isnan(self.xyz[:, 2]) == False]), jnp.max(self.xyz[:, 1][jnp.isnan(self.xyz[:, 2]) == False])])
        if save and (name != ""):
            fig.savefig("")
        if show:
            plt.show()
        return fig, ax
    def __len__(self):
        return len(self.xyz[:, 0].ravel())
    def _tree_flatten(self):
        children = (self.xyz, self.x, self.y, self.P)
        aux = {
            "power": self.power, "waist": self.waist, "res": self.resolution, 
            "chop": self.chop, "chop_threshold": self.chop_threshold, 
            "wavelen": self.wl, "mu_x": self.mu_x, "mu_y": self.mu_y, 
            "z": self.z, "mode": self.mode, "mode_type": self.mode_type
        }
        return (children, aux)
    @classmethod
    def _tree_unflatten(cls, aux, children):
        obj = cls.__new__(cls)
        obj.power = aux["power"]
        obj.waist = aux["waist"]
        obj.resolution = aux["res"]
        obj.chop = aux["chop"]
        obj.chop_threshold = aux["chop_threshold"]
        obj.wl = aux["wavelen"]
        obj.mode = aux["mode"]
        obj.mode_type = aux["mode_type"]
        obj.mu_x = aux["mu_x"]
        obj.mu_y = aux["mu_y"]
        obj.z = aux["z"]
        obj.xyz, obj.x, obj.y, obj.P = children
        return obj

tree_util.register_pytree_node(laser, laser._tree_flatten, laser._tree_unflatten)

I'm actually quite pleased with the way that this code has turned out. It took a while to familiarize myself with the `jax` package, but it seems to make significant leaps in the computation time needed for generating the laser profile and displacing the laser by some dy. I've officially replaced the original `laser.Laser` class with this new class written above. I also, arbitrarily, changed the name of the class to `laser.laser` LOL.

Some benchmarks are shown below!

In [3]:
import time
import jax
import jax.numpy as jnp
from jax import jit
import laser

jax.clear_caches() # making sure I have as much VRAM as possible available for calculations.

s = time.perf_counter() # TIMER START
L = Laser(0, 0, 0, waist = 1e-4, power = 102, res = 10000, mode = [2, 1], mode_type = "l", chop = True, chop_threshold = 1e-5) # laser creation using old class
e = time.perf_counter() # TIMER END
L_time = e - s
print(f"Using the `Laser` class, one generation took:\t{e - s:.2f} seconds.") # Laser class execution time

s = time.perf_counter() # TIMER START
l = jax.block_until_ready(laser.laser(0, 0, 0, waist = 1e-4, power = 102, res = 10000, mode = [2, 1], mode_type = 1, chop = True, chop_threshold = 1e-5)) # laser creation using new class
e = time.perf_counter() # TIMER END
l_time = e - s
print(f"Using the `laser` class, one generation took:\t{l_time:.2f} seconds.") # laser class execution time

print(f"\nThis means that a single execution time took {l_time / L_time * 100 :.2f}% of the time! \rIn other words, we have gained a {L_time / l_time:.2f}x increase in computation speed.")

print("\nWe can benchmark this by running each 100x and getting some averages.\n")
jax.clear_caches()
laser.laser_generation_bench(100, 0, 0, 0, [0, 0], 0, 10000, 1e-3, 100, True, 1e-5, 1064e-9)

Using the `Laser` class, one generation took:	6.42 seconds.
Using the `laser` class, one generation took:	1.20 seconds.

In other words, we have gained a 5.34x increase in computation speed.

We can benchmark this by running each 100x and getting some averages.

warming up by generating an initial laser...
warmed up. starting benchmark...


Total loop execution time: 9.6179 seconds
Average time per profile:  0.0962 seconds


warming up by generating an initial Laser...
warmed up. starting benchmark...


Total loop execution time: 428.4723 seconds
Average time per profile:  4.2847 seconds



After benchmarking the laser generation, we see that it takes about 0.1 seconds to generate a laser profile with a density of 100 million points! This is at least a 44x increase in computation speed when compared to the original laser class. Next, I'd like to benchmark oscillations of the laser beam. This, in theory, should be faster than before.

In [7]:
import time
import jax
import jax.numpy as jnp
from jax import jit
import laser

jax.clear_caches() # making sure I have as much VRAM as possible available for calculations.

L = Laser(0, 0, 0, waist = 1e-4, power = 102, res = 10000, mode = [2, 1], mode_type = "l", chop = True, chop_threshold = 1e-5) # laser creation using old class
s = time.perf_counter() # TIMER START
L.dy(1e-6)
e = time.perf_counter() # TIMER END
L_time = e - s
print(f"Using the `Laser` class, Laser dy function took:\t{e - s:.2f} seconds") # Laser class execution time

l = jax.block_until_ready(laser.laser(0, 0, 0, waist = 1e-4, power = 102, res = 10000, mode = [2, 1], mode_type = 1, chop = True, chop_threshold = 1e-5)) # laser creation using new class
s = time.perf_counter() # TIMER START
jax.block_until_ready(l.dy(1e-6))
e = time.perf_counter() # TIMER END
l_time = e - s
print(f"Using the `laser` class, laser dy function took:\t{l_time:.2f} seconds") # laser class execution time
print(f"This means that the execution time took {l_time / L_time * 100 :.2f}% of the time! \rIn other words, we have gained a {L_time / l_time:.2f}x increase in computation speed.")

print("\nWe can benchmark this by running each 100x and getting some averages.\n")
jax.clear_caches()
laser.laser_dy_bench(100, 0, 0, 0, [0, 0], 0, 10000, 1e-3, 100, True, 1e-5, 1064e-9)

Using the `Laser` class, Laser dy function took:	1.06 seconds
Using the `laser` class, laser dy function took:	8.16 seconds
In other words, we have gained a 0.13x increase in computation speed.

We can benchmark this by running each 100x and getting some averages.

warming up by generating an initial laser...
warmed up. starting benchmark...


laser total loop execution time: 9.5481 seconds
Average time per profile:  0.0955 seconds

warming up by generating an initial Laser...
warmed up. starting benchmark...


Laser total loop execution time: 133.1063 seconds
Average time per profile:  1.3311 seconds



Bad pipe message: %s [b'\xc9\xc4\x0f\xabe\x85\xd7j\xd7}\x99}\x0e\xf6\xa2\xf1\xd4\xba\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c']


Although the single execution took more time in the single function runs, it's seen that using the jax function in benchmarking greatly increases the efficiency of laser oscillation over 100 samples. We see at least a 13x increase in computation speed.

### Fiber Jaxification